# Data Sorting and Loading
Load and process both JSON and Parquet files from the repository.

In [ ]:
import json
import pandas as pd
import os

## Loading JSON Files

In [ ]:
# List of JSON files
json_files = ['matches.json', 'match_ids.json', 'momentum.json']

# Load each JSON file
json_data = {}
for json_file in json_files:
    if os.path.exists(json_file):
        with open(json_file, 'r') as f:
            json_data[json_file] = json.load(f)
        print(f'Loaded {json_file}: {type(json_data[json_file])} with {len(json_data[json_file])} items')
    else:
        print(f'File not found: {json_file}')

## Loading Parquet Files

In [ ]:
# List of Parquet files
parquet_files = ['acclimatization.parquet', 'historical_placebo.parquet', 'stoppages.parquet']

# Load each Parquet file as a DataFrame
parquet_data = {}
for parquet_file in parquet_files:
    if os.path.exists(parquet_file):
        parquet_data[parquet_file] = pd.read_parquet(parquet_file)
        print(f'Loaded {parquet_file}: DataFrame with shape {parquet_data[parquet_file].shape}')
        print(f'Columns: {list(parquet_data[parquet_file].columns)}')
    else:
        print(f'File not found: {parquet_file}')

## Merge Matches and Momentum Data

In [ ]:
# Convert JSON data to DataFrames for easier merging
matches_df = pd.DataFrame(json_data['matches.json'])
momentum_df = pd.DataFrame(json_data['momentum.json'])
match_ids = json_data['match_ids.json']

print(f'Matches DataFrame shape: {matches_df.shape}')
print(f'Momentum DataFrame shape: {momentum_df.shape}')
print(f'Number of match IDs: {len(match_ids)}')
print(f'\nMatches columns: {list(matches_df.columns)}')
print(f'Momentum columns: {list(momentum_df.columns)}')

In [ ]:
# Convert match_ids to strings for consistent comparison
match_ids_str = [str(mid) for mid in match_ids]

# Filter matches and momentum to only include those in match_ids
matches_filtered = matches_df[matches_df['id'].astype(str).isin(match_ids_str)].copy()
momentum_filtered = momentum_df[momentum_df['id'].astype(str).isin(match_ids_str)].copy()

print(f'Filtered Matches: {len(matches_filtered)} matches')
print(f'Filtered Momentum: {len(momentum_filtered)} momentum records')
print(f'\nMatches in both datasets: {len(set(matches_filtered["id"]) & set(momentum_filtered["id"]))}')

In [ ]:
# Merge matches and momentum data on the 'id' field
# Use 'id' for the key and rename columns to avoid conflicts
merged_data = pd.merge(
    matches_filtered,
    momentum_filtered,
    on='id',
    suffixes=('_match', '_momentum'),
    how='inner'
)

print(f'Merged data shape: {merged_data.shape}')
print(f'\nMerged columns: {list(merged_data.columns)}')
print(f'\nFirst few rows:')
print(merged_data.head())

## Inspect Data

In [ ]:
# Display first few rows of each dataframe
for name, df in parquet_data.items():
    print(f'\n{name}:')
    print(df.head())

In [ ]:
# Display merged data summary
print(f'Merged data info:')
print(f'Shape: {merged_data.shape}')
print(f'\nData types:')
print(merged_data.dtypes)
print(f'\nFirst 3 merged records:')
merged_data.head(3)

In [ ]:
# Optional: Save the merged data to a file
# merged_data.to_csv('merged_matches_momentum.csv', index=False)
# merged_data.to_parquet('merged_matches_momentum.parquet', index=False)
# merged_data.to_json('merged_matches_momentum.json', orient='records')
print('Merged data ready for use. Uncomment above to save to file.')